# M08-01 — JSON anidado y schema legacy (extra)

[← Anterior](01-teoria.ipynb) · [Siguiente →](../M03-transformacion-datos/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

**Extra.** No bloquea M03. Lees los dumps CRM (`profiles_v1` / `v2`), aplanas, unes las dos épocas y escribes un JSON anidado de salida.

Hazlo **después** de M02-02 (ya sabes schema y `coalesce`). Si no has generado datos: `python3 scripts/generate_novashop.py`.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M08-01-json-anidado-schema.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Arranque y los dos dumps

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Celda 0 + sesión. Cuento v1 y v2 **antes** de cruzarlos. Si v2 no es 200, el generador es viejo.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `v1 100` · `v2 200`. Schema de v2 con `struct` (`profile`, `contact`, `address`, `geo`) y `array` (`orders_preview`).

**Por qué este paso.** v1 es plano (M02). v2 es el árbol. No uses `multiLine`: son JSONL.

**Si no sale.** PATH / count 0: `python3 scripts/generate_novashop.py` y vuelve a la celda.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m08")
v1 = spark.read.json(str(RAW / "profiles_v1.jsonl"))
v2 = spark.read.json(str(RAW / "profiles_v2.jsonl"))
print("v1", v1.count(), "v2", v2.count())
v2.printSchema()


### Paso 2 — Bajar el árbol a columnas

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

`coalesce` del país (address vs profile vs UNK), igual que las dos fechas de M02. `size` del array no explota filas.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `country` nulo **0**. `n_preview=0` **6**. 200 filas.

**Por qué este paso.** Si `country` nulo = 8, no pusiste el `coalesce` de `profile.country`.


In [ ]:
from pyspark.sql.functions import (
    col, coalesce, lit, size, explode, struct, to_json, when, row_number,
)
from pyspark.sql.window import Window

v2_flat = v2.select(
    col("customer_id"),
    col("profile.contact.full_name").alias("full_name"),
    coalesce(
        col("profile.contact.address.country"),
        col("profile.country"),
        lit("UNK"),
    ).alias("country"),
    col("profile.contact.email.work").alias("email_work"),
    col("profile.contact.address.city").alias("city"),
    size(col("orders_preview")).alias("n_preview"),
    lit("v2").alias("feed"),
)
print("country nulo", v2_flat.where(col("country").isNull()).count())
print("n_preview=0", v2_flat.where(col("n_preview") == 0).count())
v2_flat.show(3, truncate=False)


## Prueba tú — explode vs size

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Cuenta filas con `explode("orders_preview")` y compáralo con `v2.count()`. Luego prueba `explode_outer`.

**Qué tienes que ver.** `explode`: distinct clientes **194** (se caen 6). `explode_outer`: distinct **200**. El `size` del paso 2 no cambia el count de clientes.


In [ ]:
from pyspark.sql.functions import explode_outer

inner_e = v2.select("customer_id", explode("orders_preview").alias("item"))
outer_e = v2.select("customer_id", explode_outer("orders_preview").alias("item"))
print("explode", inner_e.count(), "distinct clientes", inner_e.select("customer_id").distinct().count())
print("explode_outer", outer_e.count(), "distinct", outer_e.select("customer_id").distinct().count())


### Paso 3 — Aplanar v1 al **mismo** contrato

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

`fullName` → `full_name`. País `""` → nulo. `city` y `n_preview` no existen en 2023: nulo y 0. Columna `feed=v1`.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** 100 filas. Mismos nombres de columna que `v2_flat` (si no, el union chilla).

**Por qué este paso.** Sin `cast` en `city`, el union puede fallar por tipos.


In [ ]:
v1_flat = v1.select(
    col("customer_id"),
    col("fullName").alias("full_name"),
    when(col("country") == "", None).otherwise(col("country")).alias("country"),
    col("email").alias("email_work"),
    lit(None).cast("string").alias("city"),
    lit(0).alias("n_preview"),
    lit("v1").alias("feed"),
)
v1_flat.printSchema()
print(v1_flat.count())


### Paso 4 — Unir épocas y quedarte con una ficha

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

`unionByName` respeta nombres, no el orden. En el solape (50 ids) gana **v2**.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** **300** → **250**. 50 fichas solo-v1, 200 de v2 (el solape se queda con el árbol 2024).

**Por qué este paso.** Si usas `union` clásico y el orden de columnas no coincide, el `email` se mete en `city`.


In [ ]:
bruto = v2_flat.unionByName(v1_flat)
print("bruto", bruto.count())  # 300
w = Window.partitionBy("customer_id").orderBy(col("feed").desc())
profiles = (
    bruto.withColumn("rn", row_number().over(w))
    .where(col("rn") == 1)
    .drop("rn")
)
print("únicos", profiles.count())  # 250
print("feed v1", profiles.where(col("feed") == "v1").count())  # 50
print("feed v2", profiles.where(col("feed") == "v2").count())  # 200


## Prueba tú — ¿Y si gana v1?

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Cambia el `orderBy` a `col("feed").asc()` (v1 primero). Cuenta `feed==v2` otra vez. Luego **deja otra vez desc** (v2 gana): esa es la regla de negocio del curso.

**Qué tienes que ver.** Con `asc`: v2 baja a **150**. Markdown: qué clientes perderían el árbol anidado (C0051–C0100).


In [ ]:
w_v1 = Window.partitionBy("customer_id").orderBy(col("feed").asc())
alt = bruto.withColumn("rn", row_number().over(w_v1)).where(col("rn") == 1)
print("si gana v1, filas v2", alt.where(col("feed") == "v2").count())  # 150 (C0101–C0250)
print("regla del curso (gana v2)", profiles.where(col("feed") == "v2").count())  # 200


### Paso 5 — Escribir el documento enriquecido

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Parquet plano en staging (para Spark). JSON anidado en curated (para la app / un import a Mongo). `overwrite` para poder re-ejecutar.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `parquet` y `json` **250**. El schema releído vuelve a tener `struct`. Una app leería esos JSON; Mongo, un `mongoimport` del directorio.

**Por qué este paso.** Spark escribe un **directorio** de JSON, no un único `.json` bonito. Es normal.


In [ ]:
from paths import ensure_dirs

ensure_dirs()
profiles.write.mode("overwrite").parquet(str(STAGING / "profiles_flat"))

nested = profiles.select(
    "customer_id",
    struct(
        struct(
            col("full_name"),
            struct(col("email_work").alias("work")).alias("email"),
            struct(col("city"), col("country")).alias("address"),
        ).alias("contact"),
        struct(col("n_preview").alias("preview_orders")).alias("stats"),
    ).alias("profile"),
    col("feed"),
    lit("novashop.profile.v2").alias("schema_id"),
)
out = CURATED / "profiles_nested"
nested.write.mode("overwrite").json(str(out))
re = spark.read.json(str(out))
print("parquet", spark.read.parquet(str(STAGING / "profiles_flat")).count())
print("json", re.count())
re.printSchema()
re.select("customer_id", "profile.contact.address.country").show(3)


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

v1=100, v2=200, únicos=250. `country` nulo 0 tras coalesce.
JSON relído: 250 y `profile.contact` existe. Markdown: por qué 300 ≠ 250.


## Mejora — Clientes UNK

Cuenta `country == "UNK"` en `profiles`. ¿Vienen de v1, de v2 o de los dos? Markdown con el `feed`.

Si te atasca, el código está en la celda siguiente.


In [ ]:
unk = profiles.where(col("country") == "UNK")
print("UNK", unk.count())
unk.groupBy("feed").count().show()


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| v2 count ≠ 200 | Generador antiguo | `python3 scripts/generate_novashop.py` |
| union AnalysisException | Nombres/tipos distintos | Aplana v1 al mismo contrato; `city` con `cast` |
| 250 no sale | No filtraste rn==1 | Window por customer_id |
| explode = 200 | Array vacío o no explotaste | Casi todos tienen 1–3 items; tiene que subir |
| Un único .json | Spark escribe carpeta | Lee con `spark.read.json(dir)` |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M03 — o sigue el extra y vuelve al pipeline](../M03-transformacion-datos/01-teoria.ipynb).
